# Bulk Kelp detection in many images

This example notebook walks through how to install the latest version of kelp-o-matic, and then run it on a set of images in sequence
to effectively process a backlog of images through kelp-o-matic.

In [ ]:
!pip install git+https://github.com/HakaiInstitute/kelp-o-matic@dev

In [ ]:
from habitat_mapper import model_registry

In [ ]:
# We can see all the available models and versions available using the list_models method

model_registry.list_models()

In [ ]:
# We can get the latest revision of the kelp-rgb as follows:
# model = model_registry["kelp-rgb"]

# Or select a specific revision from the list
model = model_registry[("kelp-rgb", "20250728")]

In [ ]:
from pathlib import Path

# When can get all the tif images in a directory as follows
image_paths = list(Path("../testdata").glob("*.tif"))

# Or, we can just create the list manually
# image_paths = [
#     "/path/to/kelp_scene_1234.tif",
#     "/path/to/kelp_scene_some_other_area.tif",
#     "/path/to/additional/kelp_scene_5555.tif",
# ]

# You can check the paths all exist before running the big batch job
for p in image_paths:
    assert Path(p).is_file(), f"{p} doesn't exist or is not a file."

print(f"Found {len(image_paths)} images to process")

In [ ]:
crop_size = 2048

for i, image_path in enumerate(image_paths):
    # Modify the input path a bit to create an informative output path
    output_path = image_path.with_name(f"{image_path.stem}_kelp_rgb_hab_v{model.revision}_cs{crop_size}.tif")

    print(f"Processing image {i + 1}/{len(image_paths)}")
    print(f"{image_path.name} -> {output_path.name}")

    model.process(
        img_path=image_path,
        output_path=output_path,
        batch_size=1,  # Process just one tile at a time.
        crop_size=crop_size,  # Increase from the default 1024. Use larger is you can
        blur_kernel_size=5,  # A little post-processing blur smoothes out some label noise
        morph_kernel_size=0,  # Shut off morphological operations in post processing
        band_order=[1, 2, 3],  # RGB ordering
    )